In [2]:
# Data science helpers
from pathlib import Path
import pandas as pd
import numpy as np

import featuretools as ft

# Useful for showing multiple outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# make some folders we will need if they do not exist
Path("./data/churn").mkdir(parents=True, exist_ok=True)

In [11]:
# Read in all data
members = pd.read_csv('https://raw.githubusercontent.com/springboard-curriculum/featuretools/master/data/members.csv',
                      parse_dates=['registration_init_time'],
                      infer_datetime_format = True,
                      dtype = {'gender': 'category'})

transactions = pd.read_csv('https://raw.githubusercontent.com/springboard-curriculum/featuretools/master/data/transactions.csv',
                   parse_dates=['transaction_date', 'membership_expire_date'],
                    infer_datetime_format = True)

logs = pd.read_csv(f'https://raw.githubusercontent.com/springboard-curriculum/featuretools/master/data/logs.csv', parse_dates = ['date'])

cutoff_times = pd.read_csv(f'https://raw.githubusercontent.com/springboard-curriculum/featuretools/master/data/MS-31_labels.csv', parse_dates = ['cutoff_time'])

C:\Users\cococ\AppData\Local\Temp\ipykernel_27636\3237263135.py:2: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  members = pd.read_csv('https://raw.githubusercontent.com/springboard-curriculum/featuretools/master/data/members.csv',
C:\Users\cococ\AppData\Local\Temp\ipykernel_27636\3237263135.py:7: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  transactions = pd.read_csv('https://raw.githubusercontent.com/springboard-curriculum/featuretools/master/data/transactions.csv',


In [12]:
import featuretools as ft
import woodwork as ww

# Make entityset
es = ft.EntitySet(id = 'customers')

In [13]:
members.head()

,msno,city,bd,gender,registered_via,registration_init_time
0,8hW4+CV3D1oNM0CIsA39YljsF8M3m7g1LAX6AQd3C8I=,4,24,male,3,2014-11-04
1,yhcODfebyTYezE6KAPklcV1us9zdOYJ+7eHS7f/xgoU=,8,37,male,9,2007-02-11
2,sBlgSL0AIq49XsmBQ2KceKZNUyIxT1BwSkN/xYQLGMc=,15,21,male,3,2013-02-08
3,Xy3Au8sZKlEeHBQ+C7ro8Ni3X/dxgrtmx0Tt+jqM1zY=,1,0,NaN,9,2015-02-01
4,NiCu2GVWgT5QZbI85oYRBEDqHUZbzz2azS48jvM+khg=,12,21,male,3,2015-02-12


In [14]:
es = ft.EntitySet(id="customers")

es = es.add_dataframe(
    dataframe_name="members",
    dataframe=members,
    index="msno"
)

es = es.add_dataframe(
    dataframe_name="transactions",
    dataframe=transactions,
    index="transactions_index",
    time_index="transaction_date"
)

es = es.add_dataframe(
    dataframe_name="logs",
    dataframe=logs,
    index="logs_index",
    time_index="date"
)

C:\Users\cococ\anaconda3\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
C:\Users\cococ\anaconda3\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
C:\Users\cococ\anaconda3\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1733: UserWarning: index transactions_index not found in dataframe, creating new integer colu

In [15]:
es = es.add_relationship("members", "msno", "transactions", "msno")
es = es.add_relationship("members", "msno", "logs", "msno")


C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:393: UserWarning: Logical type Categorical for child column msno does not match parent column msno logical type Unknown. Changing child logical type to match parent.
  warnings.warn(


In [16]:
cutoff_times = cutoff_times.rename(columns={"cutoff_time": "time"})

In [17]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='members',
    cutoff_time=cutoff_times.head(100),  # prevents crash
    agg_primitives=['sum'],
    trans_primitives=[],
    training_window="30d",
    max_depth=2
)

C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1443: UserWarning: Using training_window but last_time_index is not set for dataframe transactions
  warnings.warn(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1443: UserWarning: Using training_window but last_time_index is not set for dataframe logs
  warnings.warn(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1443: UserWarning: Using training_window but last_time_index is not set for dataframe transactions
  warnings.warn(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1443: UserWarning: Using training_window but last_time_index is not set for dataframe logs
  warnings.warn(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function sum at 0x0000015460F1EA20> is currently using SeriesGroupBy.sum. In a future ve

In [18]:
feature_matrix.shape

(100, 20)

In [19]:
[col for col in feature_matrix.columns if "logs.num_unq" in col]

['SUM(logs.num_unq)']

In [20]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='members',
    cutoff_time=cutoff_times.head(1000),  # increase
    agg_primitives=['sum'],
    trans_primitives=[],
    training_window="30d",
    max_depth=2
)

C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1443: UserWarning: Using training_window but last_time_index is not set for dataframe transactions
  warnings.warn(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\entityset\entityset.py:1443: UserWarning: Using training_window but last_time_index is not set for dataframe logs
  warnings.warn(
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function sum at 0x0000015460F1EA20> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  ).agg(to_agg)
C:\Users\cococ\anaconda3\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function sum at 0x0000015460F1EA20> is currently using SeriesGroupBy.sum. In a future version of pan